# 05 · E4: lr ajustado al batch, precisión mixta y lr de la cabeza (D23 a D25)
Variantes, ambas con N = 100 y la configuración D23; cada una cambia un solo factor respecto de B:
- **B_amp**: precisión mixta (control de H7).
- **C_head**: lr 1e-4 en la cabeza de clases (H8).
Referencia: corrida E4-B en `results/e3_e4/`.

In [ ]:
import os, sys, subprocess, time, json
import warnings; warnings.filterwarnings('ignore')
REPO_URL = 'https://github.com/OscarLauLu/DETR-1INF52.git'
BRANCH   = 'rodrigo_dev'
REPO_DIR = 'DETR-1INF52'
if not os.path.exists('../src/train_e3.py'):
    if not os.path.exists(f'{REPO_DIR}/src'):
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    os.chdir(f'{REPO_DIR}/notebooks')
    !pip -q install -r ../requirements.txt
sys.path.insert(0, '../src')
from common import *
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42; set_seed(SEED)
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(env_report('../results/entorno_e4.txt'))

In [ ]:
# Evidencia: último commit del design log (debe ser ANTERIOR a las corridas de abajo)
!git -C .. log -1 --format='%h  %ci  %s' -- DESIGN_LOG.md
!git -C .. log -1 --format='%h  %ci  %s' -- src/train_e3.py

In [ ]:
import glob
# Ruta del dataset: Kaggle no siempre lo monta en el mismo lugar; se busca la carpeta de train
cands = sorted(glob.glob('/kaggle/input/**/VisDrone2019-DET-train', recursive=True))
assert cands, 'Dataset no montado: Add Input -> banuprasadb/visdrone-dataset'
ROOT = os.path.dirname(cands[0]); print('ROOT =', ROOT)
TRAIN_IMG, TRAIN_LBL = f'{ROOT}/VisDrone2019-DET-train/images', f'{ROOT}/VisDrone2019-DET-train/labels'
VAL_IMG = f'{ROOT}/VisDrone2019-DET-val/images'
N_TRAIN = 1500
TRAIN_FULL, TRAIN_JSON, VAL_JSON = '/tmp/train_full.json', '../data/train_e3.json', '../data/val_sub.json'
OUT, CACHE = '../results/e4', '/tmp/e3cache'
os.makedirs(OUT, exist_ok=True)
t = time.time()
if not os.path.exists(TRAIN_JSON):
    print('conversión de train completo:', yolo_to_coco(TRAIN_IMG, TRAIN_LBL, TRAIN_FULL), f'{time.time() - t:.0f} s')
    print('subconjunto:', make_subset(TRAIN_FULL, N_TRAIN, TRAIN_JSON, seed=SEED))
for name, js in (('train_e3', TRAIN_JSON), ('val', VAL_JSON)):
    c = np.array(list(objects_per_image(js).values()))
    print(f'{name}: {len(c)} imágenes, {c.sum()} cajas, {c.mean():.1f} por imagen, '
          f'{(c > 100).sum()} con n > 100 ({(c > 100).mean():.1%}), máximo {c.max()}')

In [ ]:
# D22 · Preprocesamiento único (D18), guardado como fp16 en disco y leído con memmap por los dos procesos
t = time.time()
if not os.path.exists(f'{CACHE}/val_labels.pt'):
    !python ../src/train_e3.py prep --train_img {TRAIN_IMG} --train_json {TRAIN_JSON} --val_img {VAL_IMG} --val_json {VAL_JSON} --cache {CACHE}
print(f'{time.time() - t:.0f} s')
!du -sh {CACHE}
!free -g | head -2

In [ ]:
VARIANTS = {
    'B_amp':  ['--N', '100', '--lr', '2e-5', '--lr_backbone', '2e-6', '--lr_drop', '14',
               '--eval_at', '8,14', '--amp'],
    'C_head': ['--N', '100', '--lr', '2e-5', '--lr_backbone', '2e-6', '--lr_drop', '14',
               '--eval_at', '8,14', '--lr_head', '1e-4'],
}
EPOCHS = 20

def cmd(name):
    return ['python', '-u', '../src/train_e3.py', 'train', '--cache', CACHE, '--val_json', VAL_JSON,
            '--out', f'{OUT}/{name}', '--epochs', str(EPOCHS)] + VARIANTS[name]

n_gpu = torch.cuda.device_count()
procs, gpu_of, t0 = {}, {}, time.time()
def launch(name, gpu):
    os.makedirs(f'{OUT}/{name}', exist_ok=True)
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
    procs[name] = subprocess.Popen(cmd(name), env=env, stdout=open(f'{OUT}/log_{name}.txt', 'w'),
                                   stderr=subprocess.STDOUT)
    gpu_of[name] = gpu

pending = list(VARIANTS)
while pending or any(p.poll() is None for p in procs.values()):
    busy = {gpu_of[n] for n in procs if procs[n].poll() is None}
    for gpu in range(max(1, n_gpu)):
        if pending and gpu not in busy:
            launch(pending.pop(0), gpu); busy.add(gpu)
    time.sleep(120)
    for name in procs:
        lines = [l for l in open(f'{OUT}/log_{name}.txt').read().splitlines() if l.startswith('[N')]
        print(f'{(time.time() - t0) / 60:5.1f} min | {name:7s} |', lines[-1] if lines else 'iniciando')

for name, p in procs.items():
    print(f'{name}: código de salida {p.returncode}')
    if p.returncode: print(open(f'{OUT}/log_{name}.txt').read()[-3000:])

In [ ]:
COLS = ['AP', 'AP50', 'AP_S', 'AP_M', 'AR@100', 'AR@500', 'AR_S@500']
ZS = json.load(open('../results/e3/zero_shot.json'))
RUNS = {'B (referencia, fp32)': '../results/e3_e4/B_lr2e-5/N100.json',
        **{n: f'{OUT}/{n}/N100.json' for n in VARIANTS}}
L = {m: json.load(open(p)) for m, p in RUNS.items()}
FIN = {m: l['eval']['20'] for m, l in L.items()}

print('6 clases mapeadas, época 20')
t6 = {'zero-shot': ZS['overall'], **{m: e['mapped6'] for m, e in FIN.items()}}
print(pd.DataFrame({m: {k: v[k] for k in COLS} for m, v in t6.items()}).T.round(4).to_string())

print('\nTiempo medio por época (H7)')
seg = {m: np.mean([h['seg'] for h in l['loss']]) for m, l in L.items()}
ref = seg['B (referencia, fp32)']
for m, s in seg.items():
    print(f'  {m:22s} {s:6.1f} s   ({100 * (1 - s / ref):.1f} % menos que B)')

print('\nAP por clase, 10 clases (H4 y H8: AP > 0 en las 10)')
print(pd.DataFrame({m: {c: v['AP'] for c, v in e['por_clase'].items()} for m, e in FIN.items()}).round(4).to_string())

In [ ]:
import json
OUT = '../results/e4'
NOTAS = {'B_amp': 'D23 + --amp (H7)', 'C_head': 'D23 + D25, lr_head 1e-4 (H8)'}
f = lambda x, d=3: f'{x:.{d}f}'.replace('.', ',')
for n in NOTAS:
    l = json.load(open(f'{OUT}/{n}/N100.json')); e = l['eval']['20']
    print(f"| E4-{n} | 2026-11-10 | 100 | T4 | {f(l['minutos'], 1)} | {f(l['loss'][-1]['loss'], 4)} | "
          f"{f(e['overall']['AR@500'])} | {f(e['gt100']['AR@500'])} | {f(e['overall']['AP'])} | "
          f"{NOTAS[n]}. 6 clases: AP {f(e['mapped6']['AP'])} |")

In [ ]:
!cd .. && zip -qr /kaggle/working/resultados_e4.zip results/e4 results/entorno_e4.txt && ls -lh /kaggle/working/resultados_e4.zip

In [ ]:
import glob
print(glob.glob('/kaggle/input/**/*test*', recursive=True)[:10])

In [ ]:
TD = '/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev'
!ls {TD}
!ls {TD}/images | wc -l
!ls {TD}/labels | wc -l
!head -3 "$(ls -d {TD}/labels/* | head -1)"